# Tiny M1 utility prompt diagnostic — inference only

Why did the base model answer Unknown to all ten arithmetic questions? Compare the original instruction and an arithmetic-specific instruction on the pinned base and the already-saved tiny adapter: 40 answers, zero training updates. Original pilot scores/gates remain unchanged. No reasoning traces are requested.

Use the Kaggle GPU kernel and attach HF_TOKEN in Kaggle Secrets. **Commit/push the new diagnostic code/config before running:** the first cell clones main and stops if they are missing. The old training notebook does not need to be rerun.


In [2]:
import os, subprocess, sys, tempfile
from pathlib import Path

remote_root = Path('/kaggle/working')
assert remote_root.is_dir(), 'Connect this notebook to the Kaggle GPU kernel.'
repo_dir = Path(tempfile.mkdtemp(prefix='utility-source-', dir=remote_root)) / 'repo'
subprocess.run(['git', 'clone', '--branch', 'main', 'https://github.com/DevulapalliTharun/persistent-llm-unlearning.git', str(repo_dir)], check=True)
os.chdir(repo_dir)
print('Repository commit:', subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip())
assert Path('src/evaluation/utility_diagnostic.py').is_file(), 'New diagnostic files are absent on GitHub main. Commit/push them before running this notebook.'
assert Path('configs/evaluation/utility_prompt_diagnostic_v1.json').is_file()


Cloning into '/kaggle/working/utility-source-0g_b8_yy/repo'...


Repository commit: 5d244def80cd70b0407df143322d092b1a2b48e7


In [3]:
# Match the recorded tiny-run stack; clean the unused optional backend in Kaggle only.
assert Path('/kaggle/working').is_dir()
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', '-r', 'requirements/m1-tiny.txt'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchao'], check=True)
subprocess.run([sys.executable, '-c', 'import torch, transformers, peft; from peft.import_utils import is_torchao_available; assert not is_torchao_available(); assert torch.cuda.is_available(); print(torch.__version__, transformers.__version__, peft.__version__)'], check=True)


2.11.0+cu130 5.20.2 0.21.2


CompletedProcess(args=['/usr/bin/python3', '-c', 'import torch, transformers, peft; from peft.import_utils import is_torchao_available; assert not is_torchao_available(); assert torch.cuda.is_available(); print(torch.__version__, transformers.__version__, peft.__version__)'], returncode=0)

In [4]:
import json

plan_dir = Path(tempfile.mkdtemp(prefix='utility-plan-', dir=remote_root))
subprocess.run([sys.executable, '-m', 'src.evaluation.utility_diagnostic', '--config', 'configs/evaluation/utility_prompt_diagnostic_v1.json', '--output-dir', str(plan_dir), '--mode', 'plan'], check=True)
plan = json.loads((plan_dir / 'plan.json').read_text())
assert plan['planned_prediction_count'] == 40 and plan['training_updates'] == 0
print('Fixed prompts:', json.dumps(plan['system_prompts'], indent=2))


{
  "config": {
    "run_type": "utility_prompt_diagnostic_v1",
    "model_id": "Qwen/Qwen3-0.6B",
    "model_revision": "c1899de289a04d12100db370d81485cdf75e47ca",
    "seed": 42,
    "evaluation": {
      "batch_size": 16,
      "max_new_tokens": 128,
      "do_sample": false,
      "thinking_enabled": false
    },
    "artifact": {
      "repo_id": "tharundevulapalli/persistent-llm-unlearning-checkpoints",
      "revision": "07d23d8b904ab48401f9c6676d05bb85a14d9fbd",
      "path": "diagnostics/m1-tiny/m1-tiny-run-8myx_22r",
      "evidence_sha256": {
        "summary.json": "f49264fd0814c9cbea2843f6df9f69168eb8d752910c2902e9b9794cb8f0344a",
        "base_utility.jsonl": "61c192e3fdb654e8b35c1e9ed05e574b69c5ed34d0e4eea105267241fb44f88c",
        "post_utility.jsonl": "39955ee46aec0e8603de17698b390f45ee1c41abd1551d060bcfbbaa4ccff251"
      }
    }
  },
  "system_prompts": {
    "original-v2": "Answer the question using one line: Answer: <answer>. Replace <answer> with only the request

In [5]:
# Read the private adapter with a secret passed only through the child environment.
from kaggle_secrets import UserSecretsClient

evaluation_env = os.environ.copy()
evaluation_env['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
output_dir = Path(tempfile.mkdtemp(prefix='m1-utility-prompt-', dir=remote_root))
print('Evaluation outputs:', output_dir, flush=True)
try:
    subprocess.run([sys.executable, '-u', '-m', 'src.evaluation.utility_diagnostic', '--config', 'configs/evaluation/utility_prompt_diagnostic_v1.json', '--output-dir', str(output_dir), '--mode', 'evaluate'], env=evaluation_env, check=True)
finally:
    evaluation_env.pop('HF_TOKEN', None)


Evaluation outputs: /kaggle/working/m1-utility-prompt-9ypiagyk


Loading weights: 100%|██████████| 311/311 [00:00<00:00, 439.21it/s]
[transformers] The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


base/original-v2: 0/10 strict correct; 10 Unknown
base/arithmetic-v1: 7/10 strict correct; 0 Unknown
adapter/original-v2: 4/10 strict correct; 0 Unknown
adapter/arithmetic-v1: 6/10 strict correct; 0 Unknown
{
  "execution_complete": true,
  "model_executed": true,
  "training_updates": 0,
  "canonical_m1": false,
  "original_pilot_result_unchanged": true,
  "groups": {
    "base/original-v2": {
      "version": "answer-only-v2",
      "answer_matches": 0,
      "format_violations": 0,
      "token_limit_hits": 0,
      "non_year_answer_matches": 0,
      "year_answer_matches": 0,
      "by_partition_and_task": {
        "utility/arithmetic": {
          "count": 10,
          "answer_matches": 0,
          "format_violations": 0,
          "token_limit_hits": 0
        }
      },
      "abstentions": 10,
      "count": 10
    },
    "base/arithmetic-v1": {
      "version": "answer-only-v2",
      "answer_matches": 7,
      "format_violations": 0,
      "token_limit_hits": 0,
      "non

In [6]:
summary = json.loads((output_dir / 'summary.json').read_text())
assert summary['execution_complete'] and summary['training_updates'] == 0
print('Original outputs reproduced:', summary['original_v2_reproduction'])
print('Paired comparison ready:', summary['paired_comparison_ready'])
for file in sorted(output_dir.glob('*.jsonl')):
    print('\n' + file.name)
    for line in file.read_text().splitlines():
        row = json.loads(line)
        print(row['example_id'], '|', row['question'], '| expected:', row['answer'], '| got:', repr(row['raw_response']))
print('Original tiny pilot remains failed. Review these outputs before choosing another training run.')


Original outputs reproduced: {'adapter': {'different_example_ids': [], 'raw_predictions_identical': True}, 'base': {'different_example_ids': [], 'raw_predictions_identical': True}}
Paired comparison ready: True

adapter_arithmetic-v1.jsonl
utility_00 | What is 12 plus 7? | expected: 19 | got: 'Answer: 19'
utility_01 | What is 35 minus 18? | expected: 17 | got: 'Answer: 17'
utility_02 | What is 6 times 7? | expected: 42 | got: 'Answer: 42'
utility_03 | What is 48 divided by 6? | expected: 8 | got: 'Answer: 8'
utility_04 | A box has 3 apples. Add 4 apples. How many apples are there? | expected: 7 | got: 'Answer: 7'
utility_05 | There are 2 boxes with 5 pens each. How many pens are there? | expected: 10 | got: 'Answer: 20'
utility_06 | One notebook costs 3 coins. How many coins do 6 notebooks cost? | expected: 18 | got: 'Answer: 18'
utility_07 | Double 6, then subtract 5. What is the result? | expected: 7 | got: 'Answer: 12'
utility_08 | There are 20 chairs. Remove 8 and add 3. How many r

In [7]:
# Export reports only. The restored adapter stays outside the small ZIP.
import zipfile

archive = Path(str(output_dir) + '-evidence.zip')
with zipfile.ZipFile(archive, 'x', compression=zipfile.ZIP_DEFLATED) as bundle:
    for file in sorted(output_dir.iterdir()):
        if file.is_file() and file.suffix in {'.json', '.jsonl', '.txt'}:
            bundle.write(file, arcname=file.name)
print('Download from Kaggle Output:', archive)


Download from Kaggle Output: /kaggle/working/m1-utility-prompt-9ypiagyk-evidence.zip
